In [12]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
import matplotlib.pyplot as plt

print(f"XGBoost version: {xgb.__version__}")

XGBoost version: 3.2.0


### Dataset Preprocessing

In [46]:
#Eliminar sensores constantes
def eliminar_constantes(df, umbral_std=0.01):
    sensor_cols = [c for c in df.columns if c.startswith("s")]
    std = df[sensor_cols].std()
    constantes = std[std < umbral_std].index.tolist()
    print(f"Sensores eliminados por baja varianza: {constantes}")
    return df.drop(columns=constantes), constantes

#Calcular y añadir la etiqueta RUL
def add_rul_train(df, rul_max=110):
    """
    RUL real = ciclos restantes hasta el fallo.
    Se trunca a rul_max (cap lineal): permite que el modelo se centre en los ciclos 
    cercanos al fallo. No hay necesidad de predecir el fallo desde el inicio del trayecto.
    """
    max_ciclo = df.groupby("motor_id")["ciclo"].max().rename("ciclo_max")
    df = df.join(max_ciclo, on="motor_id")
    df["rul"] = (df["ciclo_max"] - df["ciclo"]).clip(upper=rul_max)
    df = df.drop(columns=["ciclo_max"])
    return df

def add_rul_test(df, rul_finales, rul_max=110):
    """
    En test, NASA proporciona cuántos ciclos quedaban al final
    del fragmento observado. Se suman los ciclos restantes y se calcula
    el RUL para cada ciclo de cada motor
    """
    rul_map = {
        motor_id + 1: rul
        for motor_id, rul in enumerate(rul_finales["rul_final"].values)
    }
    max_ciclo = df.groupby("motor_id")["ciclo"].max()
    
    def rul_para_motor(row):
        rul_final = rul_map[row["motor_id"]]
        ciclos_restantes = (max_ciclo[row["motor_id"]] - row["ciclo"]) + rul_final
        return min(ciclos_restantes, rul_max)
    
    df["rul"] = df.apply(rul_para_motor, axis=1)
    return df

def norm(train_df, test_df, feature_cols):
    """
    MinMaxScaler ajustado SOLO sobre train.
    Aplica la misma transformación a test sin re-ajustar.
    """
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_df[feature_cols] = scaler.transform(test_df[feature_cols]) 
    
    return train_df, test_df, scaler

def crear_ventanas(df, feature_cols, window_size=30):
    """
    Para cada motor, recorre sus ciclos con una ventana de tamaño window_size.
    Si el motor tiene menos ciclos que window_size, se rellena con padding
    al principio (ceros)
    
    Devuelve:
        X: (N, window_size, n_features)
        y: (N,)  — RUL en el último ciclo de cada ventana
    """
    X_list, y_list = [], []
    
    for motor_id, motor in df.groupby("motor_id"):
        datos = motor[feature_cols].values   # (ciclos, n_features)
        etiqueta = motor["rul"].values           # (ciclos,)
        n_ciclos = len(datos)
        
        if n_ciclos < window_size:
            # Padding al principio con ceros
            pad = np.zeros((window_size - n_ciclos, datos.shape[1]))
            datos = np.vstack([pad, datos])
            etiqueta = np.concatenate([np.full(window_size - n_ciclos, etiqueta[0]), etiqueta])
            n_ciclos = window_size
        
        for i in range(n_ciclos - window_size + 1):
            X_list.append(datos[i : i + window_size])
            y_list.append(etiqueta[i + window_size - 1])   # RUL del último ciclo
    
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

def crear_ventanas_test(df, feature_cols, window_size=30):
    """
    Se usa la última ventana de cada motor
    """
    X_list, y_list = [], []
    
    for motor_id, motor in df.groupby("motor_id"):
        datos = motor[feature_cols].values
        etiqueta = motor["rul"].values
        n_ciclos = len(datos)
        
        if n_ciclos >= window_size:
            X_list.append(datos[-window_size:])
        else:
            pad = np.zeros((window_size - n_ciclos, datos.shape[1]))
            ventana = np.vstack([pad, datos])
            X_list.append(ventana)
        
        y_list.append(etiqueta[-1])
    
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


In [47]:
def identificar_condicion_operacional(df, n_condiciones=6):
    """
    En FD002/FD004 hay 6 condiciones operacionales distintas.
    Las identificamos agrupando op_1, op_2, op_3 con KMeans.
    En FD001/FD003 con n_condiciones=1 asigna todo al cluster 0.
    """
    op_cols = ["op_1", "op_2", "op_3"]
    
    if n_condiciones == 1:
        df["condicion"] = 0
        return df
    
    km = KMeans(n_clusters=n_condiciones, random_state=42, n_init=10)
    df["condicion"] = km.fit_predict(df[op_cols])
    return df, km


def normalizar_por_condicion(train_df, test_df, feature_cols, n_condiciones):
    """
    Para FD002/FD004: ajusta un scaler distinto por condición operacional.
    Para FD001/FD003: normalización global (igual que antes).
    
    Esto evita que el modelo confunda variación por régimen de vuelo
    con variación por degradación.
    """
    train_df[feature_cols] = train_df[feature_cols].astype(np.float64)
    test_df[feature_cols]  = test_df[feature_cols].astype(np.float64)
    
    if n_condiciones == 1:
        scaler = MinMaxScaler()
        train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
        test_df[feature_cols]  = scaler.transform(test_df[feature_cols])
        return train_df, test_df, {0: scaler}
    
    scalers = {}
    for cond in range(n_condiciones):
        mask_train = train_df["condicion"] == cond
        mask_test = test_df["condicion"]  == cond
        
        if mask_train.sum() == 0:
            continue
            
        scaler = MinMaxScaler()
        train_df.loc[mask_train, feature_cols] = scaler.fit_transform(
            train_df.loc[mask_train, feature_cols]
        )
        if mask_test.sum() > 0:
            test_df.loc[mask_test, feature_cols] = scaler.transform(
                test_df.loc[mask_test, feature_cols]
            )
        scalers[cond] = scaler
    
    return train_df, test_df, scalers

In [48]:
def añadir_features_degradacion_rapido(df, sensor_cols, window_slope=10, suavizado=3):
    """
    Versión vectorizada — entre 20x y 50x más rápida que la versión con bucles.
    
    La clave es operar sobre columnas enteras en lugar de valor a valor:
    - delta:  una resta de arrays
    - slope:  rolling OLS vectorizado con fórmula analítica
    - accel:  diff sobre el slope suavizado
    """
    df = df.copy().sort_values(["motor_id", "ciclo"])
    result_chunks = []
    
    for motor_id, grupo in df.groupby("motor_id", sort=False):
        chunk = grupo.copy()
        n = len(chunk)
    
        for col in sensor_cols:
            serie = chunk[col].values
            chunk[f"{col}_delta"] = serie - serie[0]
 
            s = pd.Series(serie)
    
            # Precomputar denominador
            w = window_slope
            x_mean = (w - 1) / 2.0
            denom  = w * (w**2 - 1) / 12.0  # escalar, se calcula una vez
    
            # Numerador
            weights_x = np.arange(w, dtype=np.float64) - x_mean  # (w,)
    
            # rolling_apply sobre toda la serie — una sola llamada
            def slope_ventana(y_ventana):
                if len(y_ventana) < 3:
                    return 0.0
                y_c = y_ventana - y_ventana.mean()
                return np.dot(weights_x[-len(y_ventana):], y_c) / denom
    
            slope = (s.rolling(window=w, min_periods=3)
                      .apply(slope_ventana, raw=True)
                      .fillna(0.0)
                      .values)
            chunk[f"{col}_slope"] = slope
    
            # ── Aceleración: diff del slope suavizado ─────────────────────
            slope_suav = (pd.Series(slope)
                           .rolling(window=suavizado, min_periods=1)
                           .mean()
                           .values)
            chunk[f"{col}_accel"] = np.gradient(slope_suav)
    
        result_chunks.append(chunk)
    
    return pd.concat(result_chunks, ignore_index=True)

In [49]:
def load_cmapss(ruta_base, subset):
    COLS = (
    ["motor_id", "ciclo"] +
    [f"op_{i}" for i in range(1, 4)] +          # 3 configuraciones operacionales
    [f"s{i}" for i in range(1, 22)]             # 21 sensores
    )
    train_raw = pd.read_csv(f"../{ruta_base}/train_{subset}.txt", sep=r"\s+", header=None, names=COLS)
    test_raw = pd.read_csv(f"../{ruta_base}/test_{subset}.txt", sep=r"\s+", header=None, names=COLS)
    rul_raw = pd.read_csv(f"../{ruta_base}/RUL_{subset}.txt", sep=r"\s+", header=None, names=["rul_final"])
    return train_raw, test_raw, rul_raw

### Construcción de features por ventana

In [50]:
def construir_features_xgb(df, sensor_cols, window_size=30):
    """
    Para cada fila (ciclo) construye un vector de features que resume
    la ventana de los últimos window_size ciclos.

    Features por sensor:
      - Valor actual
      - Delta acumulado desde ciclo 1
      - Slope local
      - Aceleración
      - Media, std, min, max, rango sobre la ventana
      - Slope de regresión lineal sobre la ventana completa
      - Último valor menos primer valor de la ventana (tendencia corta)

    Features globales:
      - Ciclo actual (normalizado por motor)
      - Ciclo relativo (ciclo / max_ciclo del motor) → proxy de edad
      - Condición operacional
    """
    registros = []

    for motor_id, grupo in df.groupby("motor_id"):
        grupo = grupo.reset_index(drop=True)
        n_ciclos  = len(grupo)
        max_ciclo = grupo["ciclo"].max()

        for t in range(n_ciclos):
            inicio  = max(0, t - window_size + 1)
            ventana = grupo.iloc[inicio:t + 1]

            fila = {}

            # ── Features globales del ciclo ──────────────────────────────
            fila["ciclo_actual"]   = grupo.loc[t, "ciclo"]
            fila["ciclo_relativo"] = grupo.loc[t, "ciclo"] / max_ciclo
            fila["condicion"]      = grupo.loc[t, "condicion"] \
                                     if "condicion" in grupo.columns else 0

            # ── Configuración operacional actual ─────────────────────────
            for op in ["op_1", "op_2", "op_3"]:
                if op in grupo.columns:
                    fila[op] = grupo.loc[t, op]

            # ── Features por sensor ──────────────────────────────────────
            for col in sensor_cols:
                v = ventana[col].values

                # Valor actual y derivadas en t
                fila[f"{col}_actual"]  = grupo.loc[t, col]
                fila[f"{col}_delta"]   = grupo.loc[t, f"{col}_delta"] \
                                         if f"{col}_delta" in grupo.columns \
                                         else v[-1] - v[0]
                fila[f"{col}_slope"]   = grupo.loc[t, f"{col}_slope"] \
                                         if f"{col}_slope" in grupo.columns else 0
                fila[f"{col}_accel"]   = grupo.loc[t, f"{col}_accel"] \
                                         if f"{col}_accel" in grupo.columns else 0

                # Estadísticas sobre la ventana
                fila[f"{col}_mean"]    = v.mean()
                fila[f"{col}_std"]     = v.std() if len(v) > 1 else 0
                fila[f"{col}_min"]     = v.min()
                fila[f"{col}_max"]     = v.max()
                fila[f"{col}_rango"]   = v.max() - v.min()

                # Diferencia entre último y primer ciclo de la ventana
                fila[f"{col}_delta_ventana"] = v[-1] - v[0]

            fila["rul"]      = grupo.loc[t, "rul"]
            fila["motor_id"] = motor_id
            registros.append(fila)

    return pd.DataFrame(registros)

CONFIG_DATASETS = {
    "FD001": {"n_condiciones": 1, "n_fallos": 1, "cnn_filtros": 64, "lstm_units": 64}, # Subset estable 
    "FD002": {"n_condiciones": 6, "n_fallos": 1, "cnn_filtros": 64, "lstm_units": 128}, # Subset con más condiciones => más tipos de degradación temporal
    "FD003": {"n_condiciones": 1, "n_fallos": 2, "cnn_filtros": 128, "lstm_units": 64}, # Más patrones posibles de degradación
    "FD004": {"n_condiciones": 6, "n_fallos": 2, "cnn_filtros": 128, "lstm_units": 128} # Ambos
}

def preparar_dataset_xgb(ruta_base, subset, window_size=60, rul_max=110):
    n_cond = CONFIG_DATASETS[subset]["n_condiciones"]

    train_raw, test_raw, rul_test = load_cmapss(ruta_base, subset)
    train_df = add_rul_train(train_raw.copy(), rul_max)
    test_df  = add_rul_test(test_raw.copy(), rul_test, rul_max)

    # Sensores útiles
    sensor_cols = [c for c in train_df.columns if c.startswith("s")]
    constantes  = train_df[sensor_cols].std()[lambda s: s < 0.01].index.tolist()
    train_df    = train_df.drop(columns=constantes)
    test_df     = test_df.drop(columns=constantes)
    sensor_cols = [c for c in sensor_cols if c not in constantes]

    # Identificar condición operacional
    if n_cond > 1:
        train_df, km = identificar_condicion_operacional(train_df, n_cond)
        # Aplicar el mismo KMeans al test — no re-ajustar
        test_df["condicion"] = km.predict(test_df[["op_1","op_2","op_3"]])
    else:
        train_df = identificar_condicion_operacional(train_df, 1)
        test_df["condicion"] = 0

    # Features de degradación
    print(f"[{subset}] Calculando features de degradación...")
    train_df = añadir_features_degradacion_rapido(train_df, sensor_cols)
    print(train_df)
    test_df  = añadir_features_degradacion_rapido(test_df,  sensor_cols)
    

    # Construir tabla plana de features
    print(f"[{subset}] Construyendo features agregadas...")
    train_feat = construir_features_xgb(train_df, sensor_cols, window_size)
    print(train_feat)
    test_feat  = construir_features_xgb(test_df,  sensor_cols, window_size)
    

    # Para test, solo el último ciclo de cada motor
    test_feat = test_feat.groupby("motor_id").last().reset_index()

    feature_cols = [c for c in train_feat.columns
                    if c not in ["rul", "motor_id"]]

    X_train_full = train_feat[feature_cols].values.astype(np.float32)
    y_train_full = train_feat["rul"].values.astype(np.float32)
    X_test       = test_feat[feature_cols].values.astype(np.float32)
    y_test       = test_feat["rul"].values.astype(np.float32)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.15, random_state=42)

    print(f"[{subset}] Train {X_train.shape} | Val {X_val.shape} "
          f"| Test {X_test.shape}")
    print(f"[{subset}] Features totales: {len(feature_cols)}")

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), feature_cols

### Entrenamiento XGBoost

In [53]:
def nasa_score_numpy(y_true, y_pred):
    diff = y_pred - y_true
    return np.sum(np.where(diff < 0,
                           np.exp(-diff / 13.0) - 1,
                           np.exp( diff / 10.0) - 1))

def entrenar_xgb(subset, ruta_base="CMAPSSData", window_size=30):

    (X_tr, y_tr), (X_val, y_val), (X_te, y_te), feature_cols = \
        preparar_dataset_xgb(ruta_base, subset, window_size)

    # Sample weights — igual que en CNN-BiLSTM
    w_tr = np.where(y_tr < 30, 4.0, np.where(y_tr < 80, 2.0, 1.0))

    model = xgb.XGBRegressor(
        n_estimators      = 1000,
        learning_rate     = 0.05,
        max_depth         = 6,
        min_child_weight  = 5,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        gamma             = 0.1,
        reg_alpha         = 0.1,    # L1
        reg_lambda        = 1.0,    # L2
        objective         = "reg:squarederror",
        eval_metric       = "rmse",
        early_stopping_rounds = 30,
        device            = "cuda",   # GPU
        random_state      = 42,
        n_jobs            = -1
    )

    model.fit(
        X_tr, y_tr,
        #sample_weight = w_tr,
        eval_set = [(X_val, y_val)],
        verbose = 100
    )

    return model, feature_cols, (X_te, y_te)


def evaluar_xgb(model, X_test, y_test, subset):
    y_pred = model.predict(X_test)
    diff   = y_pred - y_test
    score  = np.where(diff < 0,
                      np.exp(-diff / 13.0) - 1,
                      np.exp( diff / 10.0) - 1)
    rmse_v = np.sqrt(np.mean(diff**2))
    mae_v  = np.mean(np.abs(diff))

    print(f"\n── {subset} ──────────────────────────")
    print(f"RMSE  : {rmse_v:.2f}")
    print(f"MAE   : {mae_v:.2f}")
    print(f"Score : {score.sum():.1f}")
    return y_pred

### Evaluar XGB

In [54]:
RUTA = "CMAPSSData"
resultados_xgb = {}

for subset in ["FD001", "FD002"]:#, "FD003", "FD004"]:
    print(f"\n{'='*50}\n{subset}")
    model, feats, (X_te, y_te) = entrenar_xgb(subset, ruta_base=RUTA)
    y_pred = evaluar_xgb(model, X_te, y_te, subset)
    resultados_xgb[subset] = {
        "model": model, "feats": feats,
        "y_test": y_te, "y_pred": y_pred
    }



FD001
[FD001] Calculando features de degradación...
       motor_id  ciclo    op_1    op_2   op_3      s2       s3       s4  \
0             1      1 -0.0007 -0.0004  100.0  641.82  1589.70  1400.60   
1             1      2  0.0019 -0.0003  100.0  642.15  1591.82  1403.14   
2             1      3 -0.0043  0.0003  100.0  642.35  1587.99  1404.20   
3             1      4  0.0007  0.0000  100.0  642.35  1582.79  1401.87   
4             1      5 -0.0019 -0.0002  100.0  642.37  1582.85  1406.22   
...         ...    ...     ...     ...    ...     ...      ...      ...   
20626       100    196 -0.0004 -0.0003  100.0  643.49  1597.98  1428.63   
20627       100    197 -0.0016 -0.0005  100.0  643.54  1604.50  1433.58   
20628       100    198  0.0004  0.0000  100.0  643.42  1602.46  1428.18   
20629       100    199 -0.0011  0.0003  100.0  643.23  1605.26  1426.53   
20630       100    200 -0.0032 -0.0005  100.0  643.85  1600.38  1432.14   

           s7       s8  ...  s15_accel  s17_de

### Interpretación de features

In [11]:
def plot_feature_importance(model, feature_cols, subset, top_n=20):
    """
    XGBoost permite ver exactamente qué features importan más.
    Esto es oro para la sección de discusión del paper:
    puedes argumentar qué sensores son más predictivos de degradación.
    """
    importancias = model.feature_importances_
    idx          = np.argsort(importancias)[::-1][:top_n]

    plt.figure(figsize=(10, 6))
    plt.barh(range(top_n),
             importancias[idx][::-1],
             color="steelblue", alpha=0.8)
    plt.yticks(range(top_n),
               [feature_cols[i] for i in idx][::-1],
               fontsize=9)
    plt.xlabel("Importancia (gain)")
    plt.title(f"{subset} — Top {top_n} features más predictivas")
    plt.tight_layout()
    plt.show()

    print(f"\nTop 10 features — {subset}:")
    for i in idx[:10]:
        print(f"  {feature_cols[i]:35s}  {importancias[i]:.4f}")

for subset, res in resultados_xgb.items():
    plot_feature_importance(
        res["model"], res["feats"], subset, top_n=20)

NameError: name 'resultados_xgb' is not defined